# 🇻🇳 ViMind 2.0: Huấn Luyện Nền Tảng Tri Thức (64M Architecture & Textbook Pipeline)
Notebook này được thiết kế và tối ưu hóa cho GPU Kaggle (Tesla T4 16GB VRAM):
1. **Tự động clone mã nguồn ViMind từ GitHub.**
2. **Làm giàu tri thức tiền kỳ:** Kết hợp Sách giáo khoa (Math/Science), 10.000 đầu sách tiếng Việt và Wikipedia.
3. **Kiến trúc ViMind 2.0 (64M):** 12 tầng Transformer, GQA 2:1, Context 1.024 tokens, Gradient Checkpointing.
4. **Huấn luyện tiền kỳ đa chu kỳ (Multi-epoch Pretraining):** 3 Epochs với Cosine LR Schedule toàn chu kỳ.
5. **Đảm bảo 0 lỗi:** Kiểm thử tự động trước khi nạp dữ liệu thật.

In [ ]:
# 1. Clone toàn bộ mã nguồn ViMind từ GitHub và di chuyển vào thư mục làm việc
!rm -rf /kaggle/working/vimind
!git clone https://github.com/WuKong0601/ViMind.git /kaggle/working/vimind
%cd /kaggle/working/vimind


In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip install -r requirements.txt


In [ ]:
# 3. Kiểm tra thông số phần cứng GPU & Khả năng tương thích CUDA
!nvidia-smi
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU Device: {gpu_name} (CUDA Capability {cap[0]}.{cap[1]})')


In [ ]:
# 4. Tải dữ liệu SFT, DPO và Pipeline làm giàu tri thức Sách giáo khoa / Bách khoa
!python data_pipeline/download_sft.py
!python data_pipeline/download_dpo.py
# Tải và làm sạch dữ liệu Sách giáo khoa & Sách tiếng Việt chất lượng cao
!python data_pipeline/download_textbooks_and_books.py --output_path dataset/pretrain_vi_v2.jsonl --max_textbook 100000 --max_books 10000 --max_wiki 100000


In [ ]:
# 5. Chạy bộ kiểm thử toàn diện & Kiểm tra kiến trúc ViMind 2.0 (64M)
!python trainer/test_64m_dry_run.py
!python trainer/test_pipeline.py


In [ ]:
# 6. [GIAI ĐOẠN PRE-TRAINING VIMIND 2.0 (64M)]
import os, shutil

base_model_path = None
candidates_64m = [
    '/kaggle/input/vimind-64m-checkpoint/vimind_64m_final',
    '/kaggle/input/vimind-pretrained-base/vimind_64m_final',
    'out/vimind_64m_final'
]
for p in candidates_64m:
    if os.path.exists(p) and (os.path.exists(os.path.join(p, 'model.safetensors')) or os.path.exists(os.path.join(p, 'config.json'))):
        base_model_path = p
        break

if base_model_path:
    print(f'✅ Đã tìm thấy trọng số ViMind 2.0 (64M) tại: {base_model_path}')
    if base_model_path != 'out/vimind_64m_final':
        os.makedirs('out/vimind_64m_final', exist_ok=True)
        for fname in os.listdir(base_model_path):
            src = os.path.join(base_model_path, fname)
            dst = os.path.join('out/vimind_64m_final', fname)
            if os.path.isfile(src):
                shutil.copy2(src, dst)
        base_model_path = 'out/vimind_64m_final'
    print('⚡ Bỏ qua Pre-training (đã có checkpoint 64M) và chuyển tiếp sang SFT!')
else:
    print('🚀 Bắt đầu huấn luyện tiền kỳ đa chu kỳ ViMind 2.0 (64M) trên tập dữ liệu làm giàu...')
    data_file = 'dataset/pretrain_vi_v2.jsonl' if os.path.exists('dataset/pretrain_vi_v2.jsonl') else 'dataset/pretrain_vi.jsonl'
    !python -u trainer/pretrain.py \
        --model_size 64m \
        --data_path {data_file} \
        --tokenizer_dir model \
        --save_dir out \
        --save_weight vimind_64m \
        --epochs 3 \
        --batch_size 16 \
        --accumulation_steps 8 \
        --gradient_checkpointing \
        --learning_rate 5e-4 \
        --dtype float16 \
        --log_interval 50 \
        --save_interval 1000
    base_model_path = 'out/vimind_64m_final'


In [ ]:
# 7. [GIAI ĐOẠN SFT] Tinh chỉnh chỉ dẫn từ trọng số Pre-training để biến thành Chatbot
import os, shutil
sft_exists = False
candidates_sft = [
    '/kaggle/input/vimind-sft-checkpoint',
    '/kaggle/input/vimind-sft-checkpoint/vimind_sft_final',
    'out/sft/vimind_sft_final'
]
for p in candidates_sft:
    if os.path.exists(p) and (os.path.exists(os.path.join(p, 'model.safetensors')) or os.path.exists(os.path.join(p, 'config.json'))):
        if p != 'out/sft/vimind_sft_final':
            os.makedirs('out/sft/vimind_sft_final', exist_ok=True)
            for fname in os.listdir(p):
                src = os.path.join(p, fname)
                dst = os.path.join('out/sft/vimind_sft_final', fname)
                if os.path.isfile(src):
                    shutil.copy2(src, dst)
        sft_exists = True
        print(f'✅ Đã tìm thấy SFT Checkpoint tại: {p}')
        print('⚡ Bỏ qua giai đoạn SFT (tiết kiệm ~2 giờ) và chuyển ngay sang DPO Alignment!')
        break

if not sft_exists:
    print(f'🚀 Bắt đầu SFT Fine-Tuning với trọng số nền: {base_model_path}...')
    !python -u trainer/train_sft.py \
        --data_path dataset/sft_vi.jsonl \
        --tokenizer_dir model \
        --from_pretrained {base_model_path} \
        --save_dir out/sft \
        --save_weight vimind_sft \
        --batch_size 16 \
        --accumulation_steps 4 \
        --epochs 2 \
        --learning_rate 1e-4 \
        --dtype float32 \
        --log_interval 25 \
        --save_interval 500


In [ ]:
# 8. [KIỂM THỬ THÀNH PHẨM] Trò chuyện thử với mô hình ViMind vừa được huấn luyện xong
import os
import torch
from transformers import AutoTokenizer
from model.model import ViMindForCausalLM

model_path = 'out/sft/vimind_sft_final' if os.path.exists('out/sft/vimind_sft_final') else ('out/sft/vimind_sft_step_1000' if os.path.exists('out/sft/vimind_sft_step_1000') else base_model_path)
if os.path.exists(model_path):
    tokenizer = AutoTokenizer.from_pretrained('model')
    model = ViMindForCausalLM.from_pretrained(model_path).cuda()
    model.eval()

    test_prompts = [
        'Xin chào, bạn là ai và bạn có thể giúp gì cho tôi?',
        'Thủ đô của Việt Nam là gì?',
        'Hãy nêu 3 lợi ích của việc tập thể dục mỗi ngày.',
        'Làm thế nào để học lập trình Python hiệu quả?'
    ]

    print('=' * 60)
    print(f'🎉 KẾT QUẢ TRẢ LỜI CỦA VIMIND THÀNH PHẨM ({model_path}):')
    print('=' * 60)
    for p in test_prompts:
        formatted = tokenizer.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors='pt').input_ids.cuda()
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=150,
                temperature=0.7,
                top_p=0.85,
                top_k=20,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )
        ans = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        print(f'\n👤 Người dùng: {p}')
        print(f'🤖 ViMind: {ans.strip()}')
        print('-' * 60)
else:
    print(f'⚠️ Checkpoint not found at: {model_path}')


In [ ]:
# 8. [GIAI ĐOẠN DPO] Căn chỉnh sở thích con người & Giảm thiểu ảo giác (Hoàn thiện ViMind 1.0)
import os
sft_model_path = 'out/sft/vimind_sft_final'
candidates_sft = [
    '/kaggle/input/vimind-sft-checkpoint',
    '/kaggle/input/vimind-sft-checkpoint/vimind_sft_final',
    'out/sft/vimind_sft_final'
]
for p in candidates_sft:
    if os.path.exists(p) and (os.path.exists(os.path.join(p, 'model.safetensors')) or os.path.exists(os.path.join(p, 'config.json'))):
        sft_model_path = p
        break

print(f'🚀 Bắt đầu huấn luyện DPO Alignment với trọng số SFT: {sft_model_path}...')
!python -u trainer/train_dpo.py \
    --data_path dataset/dpo_vi.jsonl \
    --model_path {sft_model_path} \
    --save_dir out/dpo \
    --save_weight vimind_dpo \
    --batch_size 8 \
    --accumulation_steps 4 \
    --epochs 1 \
    --learning_rate 2e-5 \
    --beta 0.1 \
    --max_seq_len 512 \
    --log_interval 25 \
    --save_interval 500


In [ ]:
# 9. [KIỂM THỬ THÀNH PHẨM VIMIND 1.0 (DPO FINAL)]
import os
import torch
from transformers import AutoTokenizer
from model.model import ViMindForCausalLM

model_path = 'out/dpo/vimind_dpo_final' if os.path.exists('out/dpo/vimind_dpo_final') else 'out/sft/vimind_sft_final'
if os.path.exists(model_path):
    tokenizer = AutoTokenizer.from_pretrained('model')
    model = ViMindForCausalLM.from_pretrained(model_path).cuda()
    model.eval()

    test_prompts = [
        'Xin chào, bạn là ai và bạn có thể giúp gì cho tôi?',
        'Thủ đô của Việt Nam là gì?',
        'Hãy giải thích ngắn gọn trí tuệ nhân tạo là gì?',
        'Làm thế nào để học lập trình Python hiệu quả?'
    ]

    print('=' * 60)
    print(f'🎉 KẾT QUẢ TRẢ LỜI CỦA VIMIND 1.0 THÀNH PHẨM ({model_path}):')
    print('=' * 60)
    for p in test_prompts:
        formatted = tokenizer.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors='pt').input_ids.cuda()
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=150,
                temperature=0.6,
                top_p=0.85,
                top_k=20,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )
        ans = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        print(f'\n👤 Người dùng: {p}')
        print(f'🤖 ViMind 1.0: {ans.strip()}')
        print('-' * 60)
else:
    print(f'⚠️ Checkpoint not found at: {model_path}')
